<h2>2026 NZF Impacts on Caribbean - 'activity_v0.1.ipynb'</h2>

The initial EU ETS script looked at what EU ETS costs were likely to be for African states. This notebook goes one further and introduces NZF impact costs, using Scenario 24 of DNV's CIA Task 2 work, specifically reverse engineering policy impact onto the 4th IMO GHG Study dataset using DNV-derived Cost projections per unit of maritime transport supply (US$/dwt.nm). 

Forked from '/Users/apple/repos/PubEnv/2026_eu_ets_africa/2026_eu_ets_africa_v0.1.ipynb' on 24th March 2026.

In [1]:
# Import packages
import os
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None

# 1. Profiling Merchant Fleet Activity

## 1.1 Import Clean 'voyages_foc' table

In [2]:
voyages_foc_cl_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voyages_foc_cl = pd.read_csv("/Users/apple/repos/datasets/UMAS/voyages/2018/voyages_foc_2018_cl.csv", dtype=voyages_foc_cl_d).reset_index().rename(columns={"index":"voy_idx"})
voyages_foc_cl["energy_tj"] = voyages_foc_cl["energy_tj_voy"] + voyages_foc_cl["energy_tj_stop"]
voyages_foc_cl.loc[voyages_foc_cl.source_country == "Namibia", "source_iso_2"] = "NA"
voyages_foc_cl.loc[voyages_foc_cl.dest_country == "Namibia", "dest_iso_2"] = "NA"
countries_set = list(set(list(voyages_foc_cl.source_country.unique()) + list(voyages_foc_cl.dest_country.unique())))
print("Number of countries in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers in dataset: {0}.".format(len(voyages_foc_cl.imo.unique())))
print("Total CO2 produced: {0} million tonnes.".format(np.round(voyages_foc_cl.co2_t.sum() / 1000000, 1)))
voyages_foc_cl.head(2)

Number of countries in dataset: 215.
Number of IMO numbers in dataset: 58539.
Total CO2 produced: 715.4 million tonnes.


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj
0,0,8114170,525016489,IDMUO,360,IDN,ID,Indonesia,Muntok,IDSUG,360,IDN,ID,Indonesia,Sungaigerong,2018-01-23 07:00:00,2018-01-24 03:00:00,20.0,92.0,112.0,108.700000,0.0,0.0,0.0,1.287106,0.966079,2.253185,0.0,0.0,0.0,0.0,0.0,0.0,0.054959,0.041252,4.12646,3.09725,7.223710,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,0.096211
1,1,8114170,525016489,IDSUG,360,IDN,ID,Indonesia,Sungaigerong,IDPNJ,360,IDN,ID,Indonesia,Panjang,2018-01-27 23:00:00,2018-01-30 00:00:00,49.0,184.0,233.0,1187.016667,0.0,0.0,0.0,5.990044,10.898852,16.888896,0.0,0.0,0.0,0.0,0.0,0.0,0.255775,0.465381,19.20408,34.94172,54.145799,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,0.721156


## 1.2 Introducing Vessel Specifications

In [3]:
vess_specs_c = ["imo", "ship_type_standardized", "gross_tonnage", "capacity_unit", "deadweight", "teu", "gas_cap", "passengers", "cars", "size_range_standardized", "year_of_built", "flag", "engine_type", "me_fuel"]
vess_specs = pd.read_csv("/Users/apple/repos/datasets/UMAS/vessels/vessels_2018.csv", usecols=vess_specs_c)
voys_vess_foc = pd.merge(voyages_foc_cl, vess_specs, left_on=["imo"], right_on=["imo"], how="left", indicator="_voys_vess_m")
print("Merge performance after 'left' join between 'voyages_foc_cl' and 'vess_specs': \n\n{0}\n".format(voys_vess_foc._voys_vess_m.value_counts()))
voys_vess_foc.head(2)

Merge performance after 'left' join between 'voyages_foc_cl' and 'vess_specs': 

_voys_vess_m
both          2739872
left_only           0
right_only          0
Name: count, dtype: int64



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,ship_type_standardized,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m
0,0,8114170,525016489,IDMUO,360,IDN,ID,Indonesia,Muntok,IDSUG,360,IDN,ID,Indonesia,Sungaigerong,2018-01-23 07:00:00,2018-01-24 03:00:00,20.0,92.0,112.0,108.700000,0.0,0.0,0.0,1.287106,0.966079,2.253185,0.0,0.0,0.0,0.0,0.0,0.0,0.054959,0.041252,4.12646,3.09725,7.223710,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,0.096211,General cargo,0-4999,dwt,Indonesia,1981.0,2323.0,3277.0,NaN,NaN,NaN,NaN,MSD,4,both
1,1,8114170,525016489,IDSUG,360,IDN,ID,Indonesia,Sungaigerong,IDPNJ,360,IDN,ID,Indonesia,Panjang,2018-01-27 23:00:00,2018-01-30 00:00:00,49.0,184.0,233.0,1187.016667,0.0,0.0,0.0,5.990044,10.898852,16.888896,0.0,0.0,0.0,0.0,0.0,0.0,0.255775,0.465381,19.20408,34.94172,54.145799,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,142,Asia,035,South-eastern Asia,upp_mid_income,0,0,0.721156,General cargo,0-4999,dwt,Indonesia,1981.0,2323.0,3277.0,NaN,NaN,NaN,NaN,MSD,4,both


## 1.3 Filtering for Merchant Vessel types only

...! let's also filter for vessels above 5000 GT, in-line with the regulations.

In [4]:
transport_fleet_types = pd.read_csv("/Users/apple/repos/OceansEnv/code/fleet_profile/sp_shiptype5_transport_v0.2.csv", usecols=["imo_iv_type_name"]).imo_iv_type_name.unique()
voys_vess_foc_trans = voys_vess_foc[(voys_vess_foc.ship_type_standardized.isin(transport_fleet_types))].drop(columns="voy_idx").reset_index(drop=True).reset_index().rename(columns={"index":"voy_idx"})
print("Number of voyages in 'voys_vess_foc_trans': {0} of {1} ({2}%).".format(voys_vess_foc_trans.shape[0], voys_vess_foc.shape[0], round(100 * voys_vess_foc_trans.shape[0] / voys_vess_foc.shape[0], 1)))
print("CO2 emission represented in 'voys_vess_foc_trans': {0} mt of {1} mt ({2}%).".format(round(voys_vess_foc_trans.co2_t.sum() / 1e6, 1), round(voys_vess_foc.co2_t.sum() / 1e6, 1), round(100 * voys_vess_foc_trans.co2_t.sum() / voys_vess_foc.co2_t.sum(), 1)))
voys_vess_foc_trans.head(2)

Number of voyages in 'voys_vess_foc_trans': 2172203 of 2739872 (79.3%).
CO2 emission represented in 'voys_vess_foc_trans': 669.9 mt of 715.4 mt (93.6%).


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,ship_type_standardized,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both


# 2. Introducing Policy-related Economic Impacts

This analysis considers economic impacts arising from both the IMO's proposed Net-Zero Framework and the EU ETS regulation.

## 2.1 Add NZF Compliance Costs

The output of DNV’s work for Task 2 of the CIA is given in terms ‘Cost Intensity’. Cost Intensity refers to an estimated change in costs per unit of ‘transport work’, i.e. t-nm, referring to one tonne transported across one nautical mile. Given that the dataset underpinning the 4th IMO GHG Study dataset includes vessel specification fields such as vessel deadweight tonnage, as well as the distance covered as part of each voyage, we are able to work out the total amount of ‘Transport Work’ associated with each voyage. We may then combine this dataset with the outputs of DNV’s modelling under Scenario 24 to estimate indicative voyage-level cost changes in 2030, 2040 and 2050 arising from implementation of the Net-Zero Framework. The final dataset provides indicative values of what fleet-level cost impacts are likely to be in future, assuming no major change in fleet-wide vessel activity.

### 2.1.1 Matching Vessel Classes between IMO IV and DNV

There are slightly different vessel class definitions between the IMO IV voyages dataset and DNV's Task 2 Cost Intensities dataset. This step maps DNV's vessel class categorisations onto the IMO IV voyages dataset.

In [5]:
## Introducing DNV Task 2 Vessel Class categorisations to the IMO IV Voyages dataset
# DWT-based
dnv_size_conditions_dwt = [
    
    # Bulk carriers
    (voys_vess_foc_trans.ship_type_standardized == "Bulk carrier") & (voys_vess_foc_trans.deadweight < 10000),
    (voys_vess_foc_trans.ship_type_standardized == "Bulk carrier") & (voys_vess_foc_trans.deadweight >= 10000) & (voys_vess_foc_trans.deadweight < 35000),
    (voys_vess_foc_trans.ship_type_standardized == "Bulk carrier") & (voys_vess_foc_trans.deadweight >= 35000) & (voys_vess_foc_trans.deadweight < 60000),
    (voys_vess_foc_trans.ship_type_standardized == "Bulk carrier") & (voys_vess_foc_trans.deadweight >= 60000) & (voys_vess_foc_trans.deadweight < 100000),
    (voys_vess_foc_trans.ship_type_standardized == "Bulk carrier") & (voys_vess_foc_trans.deadweight >= 100000) & (voys_vess_foc_trans.deadweight < 200000),
    (voys_vess_foc_trans.ship_type_standardized == "Bulk carrier") & (voys_vess_foc_trans.deadweight >= 200000),

    # Refrigerated bulk
    (voys_vess_foc_trans.ship_type_standardized == "Refrigerated bulk") & (voys_vess_foc_trans.deadweight < 2000),
    (voys_vess_foc_trans.ship_type_standardized == "Refrigerated bulk") & (voys_vess_foc_trans.deadweight >= 2000),

    # General cargo
    (voys_vess_foc_trans.ship_type_standardized == "General cargo") & (voys_vess_foc_trans.deadweight < 5000),
    (voys_vess_foc_trans.ship_type_standardized == "General cargo") & (voys_vess_foc_trans.deadweight >= 5000) & (voys_vess_foc_trans.deadweight < 10000),
    (voys_vess_foc_trans.ship_type_standardized == "General cargo") & (voys_vess_foc_trans.deadweight >= 10000),

    # Oil tanker
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight < 5000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 5000) & (voys_vess_foc_trans.deadweight < 10000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 10000) & (voys_vess_foc_trans.deadweight < 20000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 20000) & (voys_vess_foc_trans.deadweight < 60000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 60000) & (voys_vess_foc_trans.deadweight < 80000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 80000) & (voys_vess_foc_trans.deadweight < 120000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 120000) & (voys_vess_foc_trans.deadweight < 200000),
    (voys_vess_foc_trans.ship_type_standardized == "Oil tanker") & (voys_vess_foc_trans.deadweight >= 200000),

    # Chemical tanker
    (voys_vess_foc_trans.ship_type_standardized == "Chemical tanker") & (voys_vess_foc_trans.deadweight < 5000),
    (voys_vess_foc_trans.ship_type_standardized == "Chemical tanker") & (voys_vess_foc_trans.deadweight >= 5000) & (voys_vess_foc_trans.deadweight < 10000),
    (voys_vess_foc_trans.ship_type_standardized == "Chemical tanker") & (voys_vess_foc_trans.deadweight >= 10000) & (voys_vess_foc_trans.deadweight < 20000),
    (voys_vess_foc_trans.ship_type_standardized == "Chemical tanker") & (voys_vess_foc_trans.deadweight >= 20000),

    # Other liquids tanker
    (voys_vess_foc_trans.ship_type_standardized == "Other liquids tanker"),

    # Ro-Ro
    (voys_vess_foc_trans.ship_type_standardized == "Ro-Ro") & (voys_vess_foc_trans.deadweight < 5000),
    (voys_vess_foc_trans.ship_type_standardized == "Ro-Ro") & (voys_vess_foc_trans.deadweight >= 5000)
]

dnv_size_values_dwt = [
    # Bulk carriers
    "0-9999", "10000-34999", "35000-59999", "60000-99999", "100000-199999", "200000-",
    # Refrigerated bulk  # Adding "2000-"
    "0-1999", "0-1999",# "2000-", Setting all Reefers to be 0-1999 in size.
    # General cargo
    "0-4999", "5000-9999", "10000-", 
    # Oil tanker
    "0-4999", "5000-9999", "10000-19999", "20000-59999", "60000-79999", "80000-119999", "120000-199999", "200000-", 
    # Chemical tanker
    "0-4999",  "5000-9999", "10000-19999", "20000-",
    # Other liquids tanker
    "0-", 
    # Ro-Ro
    "0-4999", "5000-"
]


## Gross Tonnage-based
dnv_size_conditions_gt = [
    # Cruise
    (voys_vess_foc_trans.ship_type_standardized == "Cruise") & (voys_vess_foc_trans.gross_tonnage < 2000),
    (voys_vess_foc_trans.ship_type_standardized == "Cruise") & (voys_vess_foc_trans.gross_tonnage >= 2000) & (voys_vess_foc_trans.gross_tonnage < 10000),
    (voys_vess_foc_trans.ship_type_standardized == "Cruise") & (voys_vess_foc_trans.gross_tonnage >= 10000) & (voys_vess_foc_trans.gross_tonnage < 60000),
    (voys_vess_foc_trans.ship_type_standardized == "Cruise") & (voys_vess_foc_trans.gross_tonnage >= 60000) & (voys_vess_foc_trans.gross_tonnage < 100000),
    (voys_vess_foc_trans.ship_type_standardized == "Cruise") & (voys_vess_foc_trans.gross_tonnage >= 100000),

    # Ferry-RoPax
    (voys_vess_foc_trans.ship_type_standardized == "Ferry-RoPax") & (voys_vess_foc_trans.gross_tonnage < 2000),
    (voys_vess_foc_trans.ship_type_standardized == "Ferry-RoPax") & (voys_vess_foc_trans.gross_tonnage >= 2000),

    # Ferry-pax only
    (voys_vess_foc_trans.ship_type_standardized == "Ferry-pax only") & (voys_vess_foc_trans.gross_tonnage < 2000),
    (voys_vess_foc_trans.ship_type_standardized == "Ferry-pax only") & (voys_vess_foc_trans.gross_tonnage >= 2000),

    # Other
    (voys_vess_foc_trans.ship_type_standardized == "Miscellaneous - fishing"), # Miscellaneous - fishing
    (voys_vess_foc_trans.ship_type_standardized == "Offshore"), # Offshore
    (voys_vess_foc_trans.ship_type_standardized == "Service - tug"), # Service - tug
    (voys_vess_foc_trans.ship_type_standardized == "Service - other"), # Service - other
    (voys_vess_foc_trans.ship_type_standardized == "Yacht") # Yacht
]

dnv_size_values_gt = [
    # Cruise
    "0-1999", "2000-9999", "10000-59999", "60000-99999", "100000-",
    # Ferry-RoPax
    "0-1999", "2000-", 
    # Ferry-pax only
    "0-1999", "2000-", 
    # Miscellaneous - fishing; Offshore; Service - tug; Service - other; Yacht.
    "0-", "0-", "0-", "0-", "0-"
]

## TEU-, CBM- and Vehicle-based
dnv_size_conditions_other = [
    # Container
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu < 1000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 1000) & (voys_vess_foc_trans.teu < 2000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 2000) & (voys_vess_foc_trans.teu < 3000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 3000) & (voys_vess_foc_trans.teu < 5000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 5000) & (voys_vess_foc_trans.teu < 8000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 8000) & (voys_vess_foc_trans.teu < 12000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 12000) & (voys_vess_foc_trans.teu < 14500),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 14500) & (voys_vess_foc_trans.teu < 19000),
    (voys_vess_foc_trans.ship_type_standardized == "Container") & (voys_vess_foc_trans.teu >= 19000),

    # Liquefied gas tanker
    (voys_vess_foc_trans.ship_type_standardized == "Liquefied gas tanker") & (voys_vess_foc_trans.gas_cap < 50000),
    (voys_vess_foc_trans.ship_type_standardized == "Liquefied gas tanker") & (voys_vess_foc_trans.gas_cap >= 50000) & (voys_vess_foc_trans.gas_cap < 200000),
    (voys_vess_foc_trans.ship_type_standardized == "Liquefied gas tanker") & (voys_vess_foc_trans.gas_cap >= 200000),

    # Vehicle, vehicle
    (voys_vess_foc_trans.ship_type_standardized == "Vehicle"),# & (voys_vess_foc_trans.cars < 4000),
    (voys_vess_foc_trans.ship_type_standardized == "Vehicle")# & (voys_vess_foc_trans.cars >= 4000)
]

dnv_size_values_other = [
    # Container, TEU
    "0-999", "1000-1999", "2000-2999", "3000-4999", "5000-7999", "8000-11999", "12000-14500", "14500-18999", "19000-",
    # Liquefied gas tanker, CBM
    "0-49999", "50000-199999", "200000-",
    # Vehicle, vehicle
    "0-3999", "0-3999" # "4000-" - Set all to be 0-3999 as 'cars' field isn't fully complete.
]

dnv_size_conditions = dnv_size_conditions_dwt + dnv_size_conditions_gt + dnv_size_conditions_other
dnv_size_values = dnv_size_values_dwt + dnv_size_values_gt + dnv_size_values_other
voys_vess_foc_trans["Size Range"] = np.select(dnv_size_conditions, dnv_size_values, default="None")

# Age Range
dnv_size_conditions = [
    (voys_vess_foc_trans.year_of_built < 2010), 
    (voys_vess_foc_trans.year_of_built >= 2010) & (voys_vess_foc_trans.year_of_built < 2020),
    (voys_vess_foc_trans.year_of_built >= 2020) & (voys_vess_foc_trans.year_of_built < 2030),
    (voys_vess_foc_trans.year_of_built >= 2030) & (voys_vess_foc_trans.year_of_built < 2040),
    (voys_vess_foc_trans.year_of_built >= 2040)
]

dnv_size_values = [
    "Build year before 2010", 
    "Build year 2010-2019", 
    "Build year 2020-2029", 
    "Build year 2030-2039",
    "Build year after 2039"
]

voys_vess_foc_trans["Age Range"] = np.select(dnv_size_conditions, dnv_size_values, default="None")
voys_vess_foc_trans = voys_vess_foc_trans.rename(columns={"ship_type_standardized": "Vessel Type"})
voys_vess_foc_trans.head(2)

,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,Size Range,Age Range
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,35000-59999,Build year before 2010
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,35000-59999,Build year before 2010


### 2.1.2 Reading Cost Intensity Changes

In [6]:
print("BAU (Low Growth) Scenario:")
BAULG_c = ["Year", "Vessel Type", "Unit", "Size Range", "Age group", "Total cost, excl pool ex. (USD)", "Transport work capacity (dwt-mile)"]
BAULG_r = {"Age group":"Age Range", "Total cost, excl pool ex. (USD)":"Costs USD", "Transport work capacity (dwt-mile)":"Work Capacity dwtnm"}
BAULG = pd.read_csv("/Users/apple/repos/datasets/DNV CIA Task 2/2026 NZF Impacts (S24)/BAULG.csv", usecols=BAULG_c)[BAULG_c].rename(columns=BAULG_r)
display(BAULG.head(2))

BAULG_g = BAULG.groupby(["Year", "Vessel Type", "Unit", "Size Range"]).agg({"Costs USD":"sum", "Work Capacity dwtnm": "sum"}).reset_index()
BAULG_g["USD per dwtnm"] = BAULG_g["Costs USD"] / BAULG_g["Work Capacity dwtnm"]
display(BAULG_g.head(2))

BAULG_p_c = [
    "Vessel Type", "Unit", "Size Range", "bau_usd_per_dwtnm_23", "bau_usd_per_dwtnm_30", "bau_usd_per_dwtnm_40", "bau_usd_per_dwtnm_50"]
BAULG_p = pd.DataFrame(BAULG_g.drop(columns=["Costs USD", "Work Capacity dwtnm"]).pivot(index=["Vessel Type", "Unit", "Size Range"], columns=["Year"]).reset_index().values, columns=BAULG_p_c)
display(BAULG_p.head(2))

BAU (Low Growth) Scenario:


,Year,Vessel Type,Unit,Size Range,Age Range,Costs USD,Work Capacity dwtnm
0,2023,Bulk carrier,dwt,0-9999,Build year 2010-2019,310713940,15709142210
1,2023,Bulk carrier,dwt,0-9999,Build year 2020-2029,67166917,3992423934


,Year,Vessel Type,Unit,Size Range,Costs USD,Work Capacity dwtnm,USD per dwtnm
0,2023,Bulk carrier,dwt,0-9999,999199048,48948130783,0.020413
1,2023,Bulk carrier,dwt,10000-34999,8132334295,1750671341305,0.004645


,Vessel Type,Unit,Size Range,bau_usd_per_dwtnm_23,bau_usd_per_dwtnm_30,bau_usd_per_dwtnm_40,bau_usd_per_dwtnm_50
0,Bulk carrier,dwt,0-9999,0.020413,0.020374,0.018795,0.018195
1,Bulk carrier,dwt,10000-34999,0.004645,0.004683,0.00389,0.003949


In [7]:
print("Scenario 24 (GFI Flex Only):")
S24_c = ["Year", "Vessel Type", "Unit", "Size Range", "Age group", "Total cost, excl pool ex. (USD)", "Transport work capacity (dwt-mile)"]
S24_r = {"Age group":"Age Range", "Total cost, excl pool ex. (USD)":"Costs USD", "Transport work capacity (dwt-mile)":"Work Capacity dwtnm"}
S24 = pd.read_csv("/Users/apple/repos/datasets/DNV CIA Task 2/2026 NZF Impacts (S24)/S24.csv", usecols=S24_c)[S24_c].rename(columns=S24_r)
display(S24.head(2))

S24_g = S24.groupby(["Year", "Vessel Type", "Unit", "Size Range"]).agg({"Costs USD":"sum", "Work Capacity dwtnm": "sum"}).reset_index()
S24_g["USD per dwtnm"] = S24_g["Costs USD"] / S24_g["Work Capacity dwtnm"]
display(S24_g.head(2))

S24_p_c = [
    "Vessel Type", "Unit", "Size Range", "s24_usd_per_dwtnm_23", "s24_usd_per_dwtnm_30", "s24_usd_per_dwtnm_40", "s24_usd_per_dwtnm_50"]
S24_p = pd.DataFrame(S24_g.drop(columns=["Costs USD", "Work Capacity dwtnm"]).pivot(index=["Vessel Type", "Unit", "Size Range"], columns=["Year"]).reset_index().values, columns=S24_p_c)
display(S24_p.head(2))

Scenario 24 (GFI Flex Only):


,Year,Vessel Type,Unit,Size Range,Age Range,Costs USD,Work Capacity dwtnm
0,2023,Bulk carrier,dwt,0-9999,Build year 2010-2019,310713940,15709142210
1,2023,Bulk carrier,dwt,0-9999,Build year 2020-2029,67166917,3992423934


,Year,Vessel Type,Unit,Size Range,Costs USD,Work Capacity dwtnm,USD per dwtnm
0,2023,Bulk carrier,dwt,0-9999,999199048,48948130783,0.020413
1,2023,Bulk carrier,dwt,10000-34999,8132334295,1750671341305,0.004645


,Vessel Type,Unit,Size Range,s24_usd_per_dwtnm_23,s24_usd_per_dwtnm_30,s24_usd_per_dwtnm_40,s24_usd_per_dwtnm_50
0,Bulk carrier,dwt,0-9999,0.020413,0.022486,0.025166,0.026146
1,Bulk carrier,dwt,10000-34999,0.004645,0.005093,0.005821,0.006452


### 2.1.3 Merge IMO IV Voyages dataset with DNV Cost Intensity Scenarios

Introduce new fields for Cost Intensities per unit of Transport Work Capacity in 2023, 2030, 2040 and 2050 under a 'Low Growth' BAU scenario and a 'GFI Flex Only' scenario from the IMO's CIA. 

In [8]:
voys_vess_foc_trans_dnv_temp = pd.merge(voys_vess_foc_trans, BAULG_p, left_on=["Vessel Type", "Size Range"], right_on=["Vessel Type", "Size Range"], how="left", indicator="_bau_temp_merge")
print("\nMerge performance between 'voys_vess_foc_trans' and 'BAULG_p': ", voys_vess_foc_trans_dnv_temp._bau_temp_merge.value_counts())

voys_vess_foc_trans_dnv = pd.merge(voys_vess_foc_trans_dnv_temp, S24_p, left_on=["Vessel Type", "Unit", "Size Range"], right_on=["Vessel Type", "Unit", "Size Range"], how="left", indicator="_bau_merge")
print("\nMerge performance between 'voys_vess_foc_trans_dnv_temp' and 'S24_p': ", voys_vess_foc_trans_dnv._bau_merge.value_counts())

# Only 'Miscellaneous - other' are merged as 'left_only'.
voys_vess_foc_trans_dnv = voys_vess_foc_trans_dnv[(voys_vess_foc_trans_dnv._bau_temp_merge == "both") & (voys_vess_foc_trans_dnv._bau_merge == "both")]
print("\nAmount of records carried forward: {0} of {1} ({2}%).".format(voys_vess_foc_trans_dnv.shape[0], voys_vess_foc_trans.shape[0], round(100 * voys_vess_foc_trans_dnv.shape[0] / voys_vess_foc_trans.shape[0], 1)))
voys_vess_foc_trans_dnv.head(2)


Merge performance between 'voys_vess_foc_trans' and 'BAULG_p':  _bau_temp_merge
both          2172143
left_only          60
right_only          0
Name: count, dtype: int64

Merge performance between 'voys_vess_foc_trans_dnv_temp' and 'S24_p':  _bau_merge
both          2172143
left_only          60
right_only          0
Name: count, dtype: int64

Amount of records carried forward: 2172143 of 2172203 (100.0%).


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,Size Range,Age Range,Unit,bau_usd_per_dwtnm_23,bau_usd_per_dwtnm_30,bau_usd_per_dwtnm_40,bau_usd_per_dwtnm_50,_bau_temp_merge,s24_usd_per_dwtnm_23,s24_usd_per_dwtnm_30,s24_usd_per_dwtnm_40,s24_usd_per_dwtnm_50,_bau_merge
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,35000-59999,Build year before 2010,dwt,0.002741,0.002845,0.002304,0.002284,both,0.002741,0.003291,0.003728,0.00394,both
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,35000-59999,Build year before 2010,dwt,0.002741,0.002845,0.002304,0.002284,both,0.002741,0.003291,0.003728,0.00394,both


### 2.1.4 Derive Total NZF Costs Associated with BAU and GFI Flexibility Mechanism Only scenarios

Achieve this through multiplication of the transport work capacity associated with each voyage and the proposed costs per unit of transport work capacity (i.e. the cost intensity).

In [9]:
# Introduce Transport Work Capacity
voys_vess_foc_trans_dnv["dwtnm"] = voys_vess_foc_trans_dnv.deadweight * voys_vess_foc_trans_dnv.dist_travelled_ais

# Evaluate Transport Costs in 2023, 2030, 2040 and 2050 under a BAULG scenario
voys_vess_foc_trans_dnv["bau_usd_23"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.bau_usd_per_dwtnm_23
voys_vess_foc_trans_dnv["bau_usd_30"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.bau_usd_per_dwtnm_30
voys_vess_foc_trans_dnv["bau_usd_40"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.bau_usd_per_dwtnm_40
voys_vess_foc_trans_dnv["bau_usd_50"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.bau_usd_per_dwtnm_50

# Evaluate Transport Costs in 2023, 2030, 2040 and 2050 under the 'GFI Flex Only' Scenario (24) of the IMO's CIA
voys_vess_foc_trans_dnv["s24_usd_23"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.s24_usd_per_dwtnm_23
voys_vess_foc_trans_dnv["s24_usd_30"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.s24_usd_per_dwtnm_30
voys_vess_foc_trans_dnv["s24_usd_40"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.s24_usd_per_dwtnm_40
voys_vess_foc_trans_dnv["s24_usd_50"] = voys_vess_foc_trans_dnv.dwtnm * voys_vess_foc_trans_dnv.s24_usd_per_dwtnm_50

# Remove unnecessary rows for clarity
dnv_cols_to_drop = [
    "Size Range", "Age Range", "Unit", 
    "bau_usd_per_dwtnm_23", "bau_usd_per_dwtnm_30", "bau_usd_per_dwtnm_40", "bau_usd_per_dwtnm_50", "_bau_temp_merge", 
    "s24_usd_per_dwtnm_23", "s24_usd_per_dwtnm_30", "s24_usd_per_dwtnm_40", "s24_usd_per_dwtnm_50", "_bau_merge"
]
voys_nzf = voys_vess_foc_trans_dnv.drop(columns=dnv_cols_to_drop)
voys_nzf.head(2)

,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,dwtnm,bau_usd_23,bau_usd_30,bau_usd_40,bau_usd_50,s24_usd_23,s24_usd_30,s24_usd_40,s24_usd_50
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,5.806091e+07,159128.795463,165190.366278,133758.79953,132626.163678,159128.795463,191097.374915,216464.89332,228787.351725
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,7.401583e+06,20285.679803,21058.406602,17051.522135,16907.134137,20285.679803,24361.02245,27594.864284,29165.72671


## 2.2 Add ETS Compliance Costs

Understand parameters of EU ETS regulation going forward which include:

<h4>Coverage Scope</h4>

In-line with the regulation, the extension of the ETS to shipping will include the following within its scope: 

<ul>
    <li>100% for intra-EU voyages</li>
    <li>50% for voyages originating outside EU ports but destined for an EU port.</li>
    <li>50% for voyages originating in the EU ports but destined for a port outside the EU.</li>
</ul>

<h4>Annual % phase-in to 2030</h4>

The extension of the EU ETS to shipping activity entails a three-year phase-in period, increasing in scope from 40% of emissions in 2024 to 70% in 2025 and 100% in 2026. It applies to cargo and passenger ships above 5000 GT from 2024 and offshore ships above 5000 GT from 2027. The EU ETS will initially cover carbon dioxide emissions and be widened to include methane and nitrous oxide from 2026. Offshore ship and general cargo ships between 400 and 5000 GT will also be required to report emissions and may be included in the EU ETS at a later stage.

<h4>ETS EUA price projections</h4>

For the EUA price, rough estimates suggest prices could be around EUR80 per tonne of CO2 in 2026. Prices to 2030 range, however most agree that the EUA price is likely to increase. Some forecasts even suggest that a 'high' price scenario could be likely, where the EUA price may exceed EUR200/USD235 (USD/EUR = 1.18) per tonne. For this reason, let's assume that the EUA price starts on the lower end of expectations in 2026 at EUR70/USD82 per tonne of CO2, however reaches this higher price of EUR200/USD235 per tonne by 2030. For the years 2023-25, the annual EUA price was observed to fluctuate between around EUR65-75/USD77-88 per tonne. Let's assume the EUA price was also EUR70/USD82 per tonne in these years 

In [10]:
# Specify Conditions and Values of ETS Phase-in
ets_phase_in_con = [
    (voys_nzf.source_region == "Europe") & (voys_nzf.dest_region == "Europe"),
    (voys_nzf.source_region != "Europe") & (voys_nzf.dest_region == "Europe"),
    (voys_nzf.source_region == "Europe") & (voys_nzf.dest_region != "Europe")]

# Specify Coverage Scope
voys_nzf["ets_coverage"] = np.select(ets_phase_in_con, [1.0, 0.5, 0.5], default=0.0)

# Calculate Costs of Compliance with EU ETS = Coverage * Phase-in * co2_t * eua_price
voys_nzf["ets_23_usd"] = voys_nzf.ets_coverage * 0.0 * voys_nzf.co2_t * 80.0 # EUR84.0 per tonne
#voys_nzf["ets_24_usd"] = voys_nzf.ets_coverage * 0.4 * voys_nzf.co2_t * 80.0 # EUR65.0 per tonne
#voys_nzf["ets_25_usd"] = voys_nzf.ets_coverage * 0.7 * voys_nzf.co2_t * 80.0 # EUR75.0 per tonne
#voys_nzf["ets_26_usd"] = voys_nzf.ets_coverage * 1.0 * voys_nzf.co2_t * 80.0 # EUR70.0 per tonne
#voys_nzf["ets_27_usd"] = voys_nzf.ets_coverage * 1.0 * voys_nzf.co2_t * 120.0
#voys_nzf["ets_28_usd"] = voys_nzf.ets_coverage * 1.0 * voys_nzf.co2_t * 160.0
#voys_nzf["ets_29_usd"] = voys_nzf.ets_coverage * 1.0 * voys_nzf.co2_t * 200.0
voys_nzf["ets_30_usd"] = voys_nzf.ets_coverage * 1.0 * voys_nzf.co2_t * 240.0
voys_nzf.head(2)

,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,dwtnm,bau_usd_23,bau_usd_30,bau_usd_40,bau_usd_50,s24_usd_23,s24_usd_30,s24_usd_40,s24_usd_50,ets_coverage,ets_23_usd,ets_30_usd
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,5.806091e+07,159128.795463,165190.366278,133758.79953,132626.163678,159128.795463,191097.374915,216464.89332,228787.351725,0.0,0.0,0.0
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,7.401583e+06,20285.679803,21058.406602,17051.522135,16907.134137,20285.679803,24361.02245,27594.864284,29165.72671,0.0,0.0,0.0


<u><h2>Output to CSV...</h2></u>

In [11]:
print("Number of voyages in 'voys_nzf': {0}.".format(voys_nzf.shape[0]))
print("CO2 emission represented in 'voys_nzf': {0} mt.".format(round(voys_nzf.co2_t.sum() / 1e6, 1)))

print("Total ETS compliance costs in 'voys_nzf' in 2030: ${0}bn".format(\
    round(voys_nzf.ets_30_usd.sum() / 1e9, 1)
))
print("Total NZF compliance costs in 'voys_nzf' in 2030: ${0}bn (+{1}%)".format(\
    round((voys_nzf.s24_usd_30.sum() - voys_nzf.bau_usd_30.sum()) / 1e9, 1),\
    round(100 * (voys_nzf.s24_usd_30.sum() - voys_nzf.bau_usd_30.sum()) / voys_nzf.bau_usd_30.sum(), 1)
))
print("Total NZF compliance costs in 'voys_nzf' in 2040: ${0}bn (+{1}%)".format(\
    round((voys_nzf.s24_usd_40.sum() - voys_nzf.bau_usd_40.sum()) / 1e9, 1),\
    round(100 * (voys_nzf.s24_usd_40.sum() - voys_nzf.bau_usd_40.sum()) / voys_nzf.bau_usd_40.sum(), 1)
))
print("Total NZF compliance costs in 'voys_nzf' in 2050: ${0}bn (+{1}%)".format(\
    round((voys_nzf.s24_usd_50.sum() - voys_nzf.bau_usd_50.sum()) / 1e9, 1),\
    round(100 * (voys_nzf.s24_usd_50.sum() - voys_nzf.bau_usd_50.sum()) / voys_nzf.bau_usd_50.sum(), 1)
))

voys_nzf.to_csv("/Users/apple/repos/CaribbEnv/2026/activity/activity_v0.1.csv", index=False)
voys_nzf.head(2)

Number of voyages in 'voys_nzf': 2172143.
CO2 emission represented in 'voys_nzf': 669.9 mt.
Total ETS compliance costs in 'voys_nzf' in 2030: $31.1bn
Total NZF compliance costs in 'voys_nzf' in 2030: $49.7bn (+16.9%)
Total NZF compliance costs in 'voys_nzf' in 2040: $167.1bn (+63.7%)
Total NZF compliance costs in 'voys_nzf' in 2050: $203.3bn (+83.2%)


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,dwtnm,bau_usd_23,bau_usd_30,bau_usd_40,bau_usd_50,s24_usd_23,s24_usd_30,s24_usd_40,s24_usd_50,ets_coverage,ets_23_usd,ets_30_usd
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,5.806091e+07,159128.795463,165190.366278,133758.79953,132626.163678,159128.795463,191097.374915,216464.89332,228787.351725,0.0,0.0,0.0
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,7.401583e+06,20285.679803,21058.406602,17051.522135,16907.134137,20285.679803,24361.02245,27594.864284,29165.72671,0.0,0.0,0.0
